In [1]:
import os
import gc
import math
import time
import torch
import pickle
import numpy as np

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.autograd as autograd

import torch.nn.functional as functional
import torch.optim.lr_scheduler as lr_scheduler

from utils import *
os.makedirs('./phi4_8x8', exist_ok=True)
torch.set_default_dtype(torch.float32) 
_ = torch.manual_seed(0)

In [2]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [3]:
if torch.cuda.is_available():
    torch_device = 'cuda'
else:
    torch_device = 'cpu'
print(f"TORCH DEVICE: {torch_device}")

TORCH DEVICE: cuda


In [4]:
dim = 256
data_filename = "phi4_8x8_lattices_for_lamda_4_mass_1_larger_and_smaller_interaction.pkl"
f = open(data_filename, 'rb')
dataset = pickle.load(f,encoding="latin1")
dataset = torch.tensor(dataset , dtype = torch.float32 , device = torch_device).reshape((-1,1,8,8))

In [5]:
dataset.shape

torch.Size([200000, 1, 8, 8])

In [21]:
batch_size = 512
no_training_samples = 100000
no_validation_samples = 25000
no_test_samples = 25000

train_dataloader = torch.utils.data.DataLoader(dataset[:100000], batch_size = 250, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(dataset[100000:125000], batch_size = batch_size, shuffle=False)

test_set = dataset[125000:150000]
test_dataloader = torch.utils.data.DataLoader(test_set, batch_size = 250, shuffle=False)

In [9]:
model = RNVP(12,(1,8,8),256,1.).to(torch_device)

In [10]:
PATH = "./phi4_8x8/model_fkl_larger_nearer_interaction.pt"
optimizer = optim.Adam(model.parameters(),lr= 10.e-6)


In [11]:
# #For resuming the checkpoint:
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

<All keys matched successfully>

In [ ]:
epochs =  25
lambda1 = 0.0 # Hyperparameter for reverse kl 
lambda2 = 1.0 # Hyperparameter for forward kl
lambda3 = 0.0 # Hyperparameter for score loss

for i in range(epochs):
    model.train()
    forward_loss = 0.
    NLL = 0.
    
    for x in train_dataloader:
        z_ , logq = model.inverse(x)
        fkl_loss = -torch.mean(logq)  # Forward Kl loss

     
        kl_loss  = fkl_loss
        optimizer.zero_grad()
        kl_loss.backward()    
        optimizer.step()
        
        forward_loss += fkl_loss.item()*(x.shape[0])
    
    model.eval()
    with torch.no_grad():
        for x in val_dataloader:
            z_, logq = model.inverse(x)
            nll = - torch.mean(logq)
            NLL += nll.item()*(x.shape[0])
            
    mean_fkl = forward_loss/no_training_samples
    # mean_rkl = reverse_loss/no_training_samples
    # mean_score = fisher_loss/no_training_samples
    # mean_total = total_loss/no_training_samples
    mean_nll = NLL/no_validation_samples

    template = 'Epoch {:3d},FKL loss: {:.4f}, Val loss: {:.4f}'
    print(template.format(i+1 , mean_fkl , mean_nll))
    # template = 'Epoch {:3d},Total loss: {:.4f},FKL loss: {:.4f},RKL loss: {:.4f}, Score loss: {:.4f}, Val loss: {:.4f}'
    # print (template.format(i+1 , mean_total , mean_fkl , mean_rkl , mean_score , mean_nll))
    

In [12]:
no_test_samples  = 25000
NLL = 0.
model.eval()
for x in test_dataloader:
    z_, logq = model.inverse(x)
    nll = - torch.mean(logq)
    NLL += nll.item()*(x.shape[0])
test_nll = NLL/no_test_samples
print(f'Test NLL: {test_nll:.4f}')

Test NLL: -5.0406


In [13]:
torch.save({'model_state_dict': model.state_dict(),'optimizer_state_dict': optimizer.state_dict(),}, PATH)

In [22]:
no_test_samples  = 25000
neg_logp = []
for x in train_dataloader:
    z_, logq = model.inverse(x)
    neg_logp.append(-logq.cpu().detach().numpy())

In [15]:
test_set.shape

torch.Size([25000, 1, 8, 8])

In [16]:
neg_logp

[array([  6.7318726 ,   6.7318726 ,   2.444664  ,   7.386612  ,
          2.6787796 ,   6.3477783 ,  -1.23629   ,   0.46627045,
          3.7313538 ,   4.642845  , -10.445801  , -13.22039   ,
        -11.838524  , -11.785393  ,  -5.93219   ,  -5.6213074 ,
        -10.849319  ,  -9.837608  ,  -7.0932693 ,  -2.9137573 ,
         -3.9539795 ,  -3.7569046 ,   8.096466  ,  -1.7983322 ,
          6.435837  ,   2.4984207 ,  -2.9422684 ,  -5.666031  ,
         -3.3725815 ,  -7.5228653 ,  -4.849663  , -10.531906  ,
         -7.5699844 ,  -8.312996  ,  -6.7617874 , -10.995705  ,
        -10.232506  , -13.123375  ,  -9.814095  ,  -4.8852615 ,
         -9.048668  ,  -7.79187   , -13.303215  ,   5.485573  ,
         -5.7585297 ,  -5.27771   ,  -9.365143  ,  -7.398979  ,
         -3.6150894 , -11.232391  , -10.321678  ,  -9.404343  ,
         -4.470909  ,   7.1837616 , -10.91716   , -10.210396  ,
         -6.5532    ,  -4.4453278 ,  -6.752556  ,  -9.375755  ,
         -9.179665  ,  -4.5613937 ,  -8.

In [23]:
data_set = dataset[:100000].cpu().detach().numpy()
neg_logp = np.array(neg_logp).reshape((-1,))

In [24]:
neg_logp.shape

(100000,)

In [25]:
data = [data_set,neg_logp]

In [26]:
output = open("Training_larger_nearer_interaction_test_data_phi4_8x8_NF_trained_for_lambda_4.pkl", 'wb')
pickle.dump(data, output)
output.close()